# 使用 Hologres 作为 OpenAI 嵌入的向量数据库

本 notebook 将一步步指导你使用 Hologres 作为 OpenAI 嵌入的向量数据库。

本 notebook 展示了一个端到端流程：
1. 使用由 OpenAI API 预先计算好的嵌入。
2. 将嵌入存储到 Hologres 云实例中。
3. 使用 OpenAI API 将原始文本查询转换为嵌入。
4. 使用 Hologres 在创建的集合中执行最近邻搜索。
5. 在提示词工程中将搜索结果作为上下文提供给大语言模型

### 什么是 Hologres

[Hologres](https://www.alibabacloud.com/help/en/hologres/latest/what-is-hologres) 是阿里云开发的一站式实时数据仓库服务。你可以使用 Hologres 实时写入、更新、处理和分析海量数据。Hologres 支持标准 SQL 语法，兼容 PostgreSQL，并支持大多数 PostgreSQL 函数。Hologres 支持对高达 PB 级数据进行联机分析处理（OLAP）和即席分析，并提供高并发、低延迟的在线数据服务。Hologres 支持多种工作负载的细粒度隔离以及企业级安全能力。Hologres 与 MaxCompute、实时计算 Flink 版（Realtime Compute for Apache Flink）和 DataWorks 深度集成，为企业提供全栈的在线与离线数据仓库解决方案。

Hologres 通过采用 [Proxima](https://www.alibabacloud.com/help/en/hologres/latest/vector-processing) 提供向量数据库功能。

Proxima 是阿里巴巴达摩院开发的高性能软件库，可用于搜索向量的最近邻。与 Facebook AI Similarity Search（Faiss）等同类开源软件相比，Proxima 具有更高的稳定性和性能。Proxima 提供了在业界性能和效果领先的基础模块，可用于搜索相似的图片、视频或人脸。Hologres 与 Proxima 深度集成，提供高性能的向量搜索服务。

### 部署选项

- [点击这里](https://www.alibabacloud.com/product/hologres)快速部署 [Hologres 数据仓库](https://www.alibabacloud.com/help/en/hologres/latest/getting-started)。


## 前提条件

为了完成本练习，我们需要准备以下几样东西：

1. Hologres 云服务器实例。
2. 用于与向量数据库交互的 'psycopg2-binary' 库。其他任何 PostgreSQL 客户端库也可以。
3. 一个 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)。



我们可以运行一个简单的 curl 命令，验证服务器是否已成功启动：


### 安装依赖

本 notebook 显然需要 `openai` 和 `psycopg2-binary` 包，但我们还会用到一些其他库。以下命令会安装所有这些库：


In [ ]:
! pip install openai psycopg2-binary pandas wget

### 准备你的 OpenAI API 密钥

OpenAI API 密钥用于对文档和查询进行向量化。

如果你还没有 OpenAI API 密钥，可以从 [https://beta.openai.com/account/api-keys](https://beta.openai.com/account/api-keys) 获取。

拿到密钥后，请将其以 `OPENAI_API_KEY` 的名称添加到环境变量中。

In [2]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.
import os

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

if os.getenv("OPENAI_API_KEY") is not None:
    print("OPENAI_API_KEY is ready")
else:
    print("OPENAI_API_KEY environment variable not found")

OPENAI_API_KEY is ready


## 连接到 Hologres
首先将连接信息添加到环境变量中，或者你也可以直接修改下面的 "psycopg2.connect" 参数

使用官方 Python 库可以轻松连接到正在运行的 Hologres 服务器实例：

In [3]:
import os
import psycopg2

# Note. alternatively you can set a temporary env variable like this:
# os.environ["PGHOST"] = "your_host"
# os.environ["PGPORT"] "5432"),
# os.environ["PGDATABASE"] "postgres"),
# os.environ["PGUSER"] "user"),
# os.environ["PGPASSWORD"] "password"),

connection = psycopg2.connect(
    host=os.environ.get("PGHOST", "localhost"),
    port=os.environ.get("PGPORT", "5432"),
    database=os.environ.get("PGDATABASE", "postgres"),
    user=os.environ.get("PGUSER", "user"),
    password=os.environ.get("PGPASSWORD", "password")
)
connection.set_session(autocommit=True)

# Create a new cursor object
cursor = connection.cursor()

我们可以运行任意可用的方法来测试连接：

In [4]:

# Execute a simple query to test the connection
cursor.execute("SELECT 1;")
result = cursor.fetchone()

# Check the query result
if result == (1,):
    print("Connection successful!")
else:
    print("Connection failed.")

Connection successful!


In [ ]:
import wget

embeddings_url = "https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip"

# The file is ~700 MB so this will take some time
wget.download(embeddings_url)

然后需要解压下载的文件：

In [6]:
import zipfile
import os
import re
import tempfile

current_directory = os.getcwd()
zip_file_path = os.path.join(current_directory, "vector_database_wikipedia_articles_embedded.zip")
output_directory = os.path.join(current_directory, "../../data")

with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
    zip_ref.extractall(output_directory)


# check the csv file exist
file_name = "vector_database_wikipedia_articles_embedded.csv"
data_directory = os.path.join(current_directory, "../../data")
file_path = os.path.join(data_directory, file_name)


if os.path.exists(file_path):
    print(f"The file {file_name} exists in the data directory.")
else:
    print(f"The file {file_name} does not exist in the data directory.")


The file vector_database_wikipedia_articles_embedded.csv exists in the data directory.


## 加载数据

在本节中，我们将加载在本次会话之前准备好的数据，这样你就不必用自己的额度重新计算维基百科文章的嵌入。

In [16]:
!unzip -n vector_database_wikipedia_articles_embedded.zip
!ls -lh vector_database_wikipedia_articles_embedded.csv

Archive:  vector_database_wikipedia_articles_embedded.zip
-rw-r--r--@ 1 geng  staff   1.7G Jan 31 01:19 vector_database_wikipedia_articles_embedded.csv


看一下数据。

In [21]:
import pandas, json
data = pandas.read_csv('../../data/vector_database_wikipedia_articles_embedded.csv')
data

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4
...,...,...,...,...,...,...,...
24995,98295,https://simple.wikipedia.org/wiki/Geneva,Geneva,"Geneva (, , , , ) is the second biggest cit...","[-0.015773078426718712, 0.01737344264984131, 0...","[0.008000412955880165, 0.02008531428873539, 0....",24995
24996,98316,https://simple.wikipedia.org/wiki/Concubinage,Concubinage,Concubinage is the state of a woman in a relat...,"[-0.00519518880173564, 0.005898841191083193, 0...","[-0.01736736111342907, -0.002740012714639306, ...",24996
24997,98318,https://simple.wikipedia.org/wiki/Mistress%20%...,Mistress (lover),A mistress is a man's long term female sexual ...,"[-0.023164259269833565, -0.02052430994808674, ...","[-0.017878392711281776, -0.0004517830966506153...",24997
24998,98326,https://simple.wikipedia.org/wiki/Eastern%20Front,Eastern Front,Eastern Front can be one of the following:\n\n...,"[-0.00681863259524107, 0.002171179046854377, 8...","[-0.0019235472427681088, -0.004023272544145584...",24998


In [26]:
title_vector_length = len(json.loads(data['title_vector'].iloc[0]))
content_vector_length = len(json.loads(data['content_vector'].iloc[0]))

print(title_vector_length, content_vector_length)

1536 1536


### 创建表和 Proxima 向量索引

Hologres 将数据存储在 __tables__（表）中，其中每个对象至少由一个向量描述。我们的表将命名为 **articles**，每个对象将同时由 **title** 和 **content** 向量描述。

我们先创建一张表，并在 **title** 和 **content** 上都创建 Proxima 索引，然后用预先计算好的嵌入填充它。

In [3]:
cursor.execute('CREATE EXTENSION IF NOT EXISTS proxima;')
create_proxima_table_sql = '''
BEGIN;
DROP TABLE IF EXISTS articles;
CREATE TABLE articles (
    id INT PRIMARY KEY NOT NULL,
    url TEXT,
    title TEXT,
    content TEXT,
    title_vector float4[] check(
        array_ndims(title_vector) = 1 and 
        array_length(title_vector, 1) = 1536
    ), -- define the vectors
    content_vector float4[] check(
        array_ndims(content_vector) = 1 and 
        array_length(content_vector, 1) = 1536
    ),
    vector_id INT
);

-- Create indexes for the vector fields.
call set_table_property(
    'articles',
    'proxima_vectors', 
    '{
        "title_vector":{"algorithm":"Graph","distance_method":"Euclidean","builder_params":{"min_flush_proxima_row_count" : 10}},
        "content_vector":{"algorithm":"Graph","distance_method":"Euclidean","builder_params":{"min_flush_proxima_row_count" : 10}}
    }'
);  

COMMIT;
'''

# Execute the SQL statements (will autocommit)
cursor.execute(create_proxima_table_sql)

### 上传数据

现在使用 [COPY 语句](https://www.alibabacloud.com/help/en/hologres/latest/use-the-copy-statement-to-import-or-export-data)将数据上传到 Hologres 云实例。根据网络带宽不同，这可能需要 5-10 分钟。

In [4]:
import io

# Path to the unzipped CSV file
csv_file_path = '../../data/vector_database_wikipedia_articles_embedded.csv'

# In SQL, arrays are surrounded by {}, rather than []
def process_file(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            # Replace '[' with '{' and ']' with '}'
            modified_line = line.replace('[', '{').replace(']', '}')
            yield modified_line

# Create a StringIO object to store the modified lines
modified_lines = io.StringIO(''.join(list(process_file(csv_file_path))))

# Create the COPY command for the copy_expert method
copy_command = '''
COPY public.articles (id, url, title, content, title_vector, content_vector, vector_id)
FROM STDIN WITH (FORMAT CSV, HEADER true, DELIMITER ',');
'''

# Execute the COPY command using the copy_expert method
cursor.copy_expert(copy_command, modified_lines)

Proxima 索引会在后台构建。在此期间我们也可以进行搜索，但没有向量索引时查询会很慢。使用以下命令等待索引构建完成。

In [5]:
cursor.execute('vacuum articles;')

In [6]:
# Check the collection size to make sure all the points have been stored
count_sql = "select count(*) from articles;"
cursor.execute(count_sql)
result = cursor.fetchone()
print(f"Count:{result[0]}")


Count:25000


## 搜索数据

数据上传完成后，我们就开始在集合中查询最接近的向量。我们可以提供额外的参数 `vector_name`，在基于标题和基于内容的搜索之间切换。由于预先计算的嵌入是用 OpenAI 的 `text-embedding-3-small` 模型创建的，因此搜索时也必须使用该模型。


In [11]:
import openai
def query_knn(query, table_name, vector_name="title_vector", top_k=20):

    # Creates embedding vector from user query
    embedded_query = openai.Embedding.create(
        input=query,
        model="text-embedding-3-small",
    )["data"][0]["embedding"]

    # Convert the embedded_query to PostgreSQL compatible format
    embedded_query_pg = "{" + ",".join(map(str, embedded_query)) + "}"

    # Create SQL query
    query_sql = f"""
    SELECT id, url, title, pm_approx_euclidean_distance({vector_name},'{embedded_query_pg}'::float4[]) AS distance
    FROM {table_name}
    ORDER BY distance
    LIMIT {top_k};
    """
    # Execute the query
    cursor.execute(query_sql)
    results = cursor.fetchall()

    return results

In [13]:
query_results = query_knn("modern art in Europe", "Articles")
for i, result in enumerate(query_results):
    print(f"{i + 1}. {result[2]} (Score: {round(1 - result[3], 3)})")

1. Museum of Modern Art (Score: 0.501)
2. Western Europe (Score: 0.485)
3. Renaissance art (Score: 0.479)
4. Pop art (Score: 0.472)
5. Northern Europe (Score: 0.461)
6. Hellenistic art (Score: 0.458)
7. Modernist literature (Score: 0.447)
8. Art film (Score: 0.44)
9. Central Europe (Score: 0.439)
10. Art (Score: 0.437)
11. European (Score: 0.437)
12. Byzantine art (Score: 0.436)
13. Postmodernism (Score: 0.435)
14. Eastern Europe (Score: 0.433)
15. Cubism (Score: 0.433)
16. Europe (Score: 0.432)
17. Impressionism (Score: 0.432)
18. Bauhaus (Score: 0.431)
19. Surrealism (Score: 0.429)
20. Expressionism (Score: 0.429)


In [14]:
# This time we'll query using content vector
query_results = query_knn("Famous battles in Scottish history", "Articles", "content_vector")
for i, result in enumerate(query_results):
    print(f"{i + 1}. {result[2]} (Score: {round(1 - result[3], 3)})")

1. Battle of Bannockburn (Score: 0.489)
2. Wars of Scottish Independence (Score: 0.474)
3. 1651 (Score: 0.457)
4. First War of Scottish Independence (Score: 0.452)
5. Robert I of Scotland (Score: 0.445)
6. 841 (Score: 0.441)
7. 1716 (Score: 0.441)
8. 1314 (Score: 0.429)
9. 1263 (Score: 0.428)
10. William Wallace (Score: 0.426)
11. Stirling (Score: 0.419)
12. 1306 (Score: 0.419)
13. 1746 (Score: 0.418)
14. 1040s (Score: 0.414)
15. 1106 (Score: 0.412)
16. 1304 (Score: 0.411)
17. David II of Scotland (Score: 0.408)
18. Braveheart (Score: 0.407)
19. 1124 (Score: 0.406)
20. July 27 (Score: 0.405)
